In [3]:
!pip install deepface opencv-python mtcnn


   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.9 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.9 MB 1.6 MB/s eta 0:00:01
   ---------------- ----------------------- 0.8/1.9 MB 1.6 MB/s eta 0:00:01
   ---------------------- ----------------- 1.0/1.9 MB 1.5 MB/s eta 0:00:01
   --------------------------------- ------ 1.6/1.9 MB 1.6 MB/s eta 0:00:01
   ---------------------------------------- 1.9/1.9 MB 1.7 MB/s  0:00:01

  Attempting uninstall: lz4

    Found existing installation: lz4 4.3.2

    Uninstalling lz4-4.3.2:

      Successfully uninstalled lz4-4.3.2

   --- ------------------------------------  1/11 [gunicorn]
   --- ------------------------------------  1/11 [gunicorn]
   --- ------------------------------------  1/11 [gunicorn]
   --- ------------------------------------  1/11 [gunicorn]
   --- ------------------------------------  1/11 [gunicorn]
   --- ----------------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import cv2
import numpy as np
from deepface import DeepFace
from google.colab.patches import cv2_imshow


In [29]:
from google.colab import files
uploaded = files.upload()


Saving WhatsApp Image 2026-02-28 at 11.30.26 AM.jpeg to WhatsApp Image 2026-02-28 at 11.30.26 AM (1).jpeg
Saving WhatsApp Image 2026-02-28 at 11.30.26 AM1.jpeg to WhatsApp Image 2026-02-28 at 11.30.26 AM1 (1).jpeg
Saving WhatsApp Image 2026-02-28 at 11.30.26 AM2.jpeg to WhatsApp Image 2026-02-28 at 11.30.26 AM2 (1).jpeg
Saving WhatsApp Image 2026-02-28 at 11.30.26 AM3.jpeg to WhatsApp Image 2026-02-28 at 11.30.26 AM3 (1).jpeg
Saving WhatsApp Image 2026-02-28 at 11.30.26 AM4.jpeg to WhatsApp Image 2026-02-28 at 11.30.26 AM4 (1).jpeg


In [30]:
import os

AUTHORIZED_DB = "authorized_db"
os.makedirs(AUTHORIZED_DB, exist_ok=True)

persons = ["CEO"]

for p in persons:
    os.makedirs(f"{AUTHORIZED_DB}/{p}", exist_ok=True)

print("Folders created!")


Folders created!


In [31]:
from google.colab import files

print("Upload 5 images of CEO")
uploaded = files.upload()

for file in uploaded.keys():
    os.rename(file, f"{AUTHORIZED_DB}/CEO/{file}")


Upload 5 images of CEO


Saving WhatsApp Image 2026-02-28 at 11.30.26 AM.jpeg to WhatsApp Image 2026-02-28 at 11.30.26 AM (2).jpeg
Saving WhatsApp Image 2026-02-28 at 11.30.26 AM1.jpeg to WhatsApp Image 2026-02-28 at 11.30.26 AM1 (2).jpeg
Saving WhatsApp Image 2026-02-28 at 11.30.26 AM2.jpeg to WhatsApp Image 2026-02-28 at 11.30.26 AM2 (2).jpeg
Saving WhatsApp Image 2026-02-28 at 11.30.26 AM3.jpeg to WhatsApp Image 2026-02-28 at 11.30.26 AM3 (2).jpeg
Saving WhatsApp Image 2026-02-28 at 11.30.26 AM4.jpeg to WhatsApp Image 2026-02-28 at 11.30.26 AM4 (2).jpeg


In [32]:
from deepface import DeepFace
import os

def check_authorized():
    live_img = "live.jpg"
    authorized = False
    person_name = "Unknown"

    for person in persons:
        folder = f"{AUTHORIZED_DB}/{person}"

        for img_file in os.listdir(folder):
            auth_img = f"{folder}/{img_file}"

            try:
                result = DeepFace.verify(
                    img1_path=live_img,
                    img2_path=auth_img,
                    model_name="Facenet512",
                    detector_backend="retinaface",
                    distance_metric="euclidean_l2",
                    enforce_detection=False
                )

                # threshold tuned for real-life images
                if result["verified"] and result["distance"] < 1.10:
                    authorized = True
                    person_name = person
                    return authorized, person_name

            except:
                pass

    return authorized, person_name


In [34]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo():
    js = Javascript('''
        async function takePhoto() {
            const video = document.createElement('video');
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            await video.play();

            await new Promise(resolve => setTimeout(resolve, 1500));

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);

            stream.getTracks().forEach(track => track.stop());
            return canvas.toDataURL('image/jpeg', 1.0);
        }
    ''')

    display(js)
    data = eval_js('takePhoto()')
    img_bytes = b64decode(data.split(',')[1])

    with open("live.jpg", "wb") as f:
        f.write(img_bytes)

    print("📸 Captured!")

take_photo()


<IPython.core.display.Javascript object>

📸 Captured!


In [35]:
authorized, name = check_authorized()

if authorized:
    print("✔ AUTHORIZED PERSON:", name)
else:
    print("❌ UNAUTHORIZED PERSON DETECTED!")


✔ AUTHORIZED PERSON: CEO


In [36]:
!pip install twilio requests

In [37]:
import base64
import requests
from twilio.rest import Client


In [39]:
# Insert your Twilio and ImgBB API keys before running this notebook.
# Twilio credentials
account_sid = os.getenv("TWILIO_SID")
auth_token = os.getenv("TWILIO_TOKEN")



client = Client(account_sid, auth_token)

# WhatsApp numbers
twilio_whatsapp_number = "whatsapp:+14155238886"   # Twilio sandbox number
your_whatsapp_number = os.getenv("WHATSAPP_NUMBER")    # Your phone number

# ImgBB API key
IMGBB_API_KEY = os.getenv("IMGBB_KEY")

In [40]:
def upload_image():

    with open("live.jpg", "rb") as file:
        encoded = base64.b64encode(file.read())

    url = "https://api.imgbb.com/1/upload"

    payload = {
        "key": IMGBB_API_KEY,
        "image": encoded
    }

    response = requests.post(url, payload)
    data = response.json()

    image_url = data["data"]["url"]

    return image_url

In [42]:
def send_whatsapp_message(message_text):

    image_url = upload_image()

    message = client.messages.create(
        body=message_text,
        from_=twilio_whatsapp_number,
        to=your_whatsapp_number,
        media_url=[image_url]
    )

    print("Message sent successfully!")

In [43]:
authorized, name = check_authorized()

if authorized:
    send_whatsapp_message(f"✅ Authorized person detected: {name}")
else:
    send_whatsapp_message("🚨 Unauthorized person detected!")

Message sent successfully!
